# RDD Visual Analysis: Per-Game Hydration Break Effects
Analyze each game separately, extract pre/post hydration trends, and visualize trend changes.

In [ ]:
import json
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## Load and Prepare Data

In [ ]:
# Load JSON files
with open('matches.json', 'r') as f:
    matches_df = pd.DataFrame(json.load(f))

with open('momentum.json', 'r') as f:
    momentum_df = pd.DataFrame(json.load(f))

with open('match_ids.json', 'r') as f:
    match_ids = json.load(f)

# Filter to match_ids
match_ids_str = [str(mid) for mid in match_ids]
matches_filtered = matches_df[matches_df['id'].astype(str).isin(match_ids_str)].copy()
momentum_filtered = momentum_df[momentum_df['id'].astype(str).isin(match_ids_str)].copy()

# Merge
merged_data = pd.merge(
    matches_filtered,
    momentum_filtered,
    on='id',
    suffixes=('_match', '_momentum'),
    how='inner'
)

print(f'Loaded {len(merged_data)} matches')

## Per-Game Analysis: Extract Hydration Breaks and Calculate Trends

In [ ]:
def extract_hydration_windows(match_data, window_size=10):
    """
    For a single match, extract all hydration breaks and surrounding momentum data.
    Returns list of windows with pre/post slopes.
    """
    match_id = match_data['id']
    series = match_data['series']
    stoppages = match_data['stoppages']
    home_team = match_data['home_match']
    away_team = match_data['away_match']
    
    # Convert series to DataFrame
    if not isinstance(series, list):
        return []
    
    momentum_series = pd.DataFrame(series, columns=['minute', 'momentum'])
    
    # Find all hydration breaks
    hydration_breaks = []
    if isinstance(stoppages, list):
        for stoppage in stoppages:
            if len(stoppage) >= 2 and stoppage[1] == 'hydration':
                hydration_breaks.append({
                    'minute': stoppage[0],
                    'duration': stoppage[2] if len(stoppage) > 2 else None
                })
    
    if len(hydration_breaks) == 0:
        return []
    
    # Extract windows for each hydration break
    windows = []
    for hydration_break in hydration_breaks:
        hyd_minute = hydration_break['minute']
        
        # Define window bounds
        lower_bound = hyd_minute - window_size
        upper_bound = hyd_minute + window_size
        
        # Extract data in window
        window_data = momentum_series[
            (momentum_series['minute'] >= lower_bound) & 
            (momentum_series['minute'] <= upper_bound)
        ].copy()
        
        if len(window_data) < 3:  # Need at least 3 points for meaningful regression
            continue
        
        # Split into before/after
        before_data = window_data[window_data['minute'] < hyd_minute]
        after_data = window_data[window_data['minute'] >= hyd_minute]
        
        # Calculate slopes
        pre_slope = None
        post_slope = None
        pre_r2 = None
        post_r2 = None
        trend_change = None
        
        if len(before_data) >= 2:
            z_before = np.polyfit(before_data['minute'], before_data['momentum'], 1)
            pre_slope = z_before[0]
            p_before = np.poly1d(z_before)
            ss_res = np.sum((before_data['momentum'] - p_before(before_data['minute']))**2)
            ss_tot = np.sum((before_data['momentum'] - before_data['momentum'].mean())**2)
            pre_r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        if len(after_data) >= 2:
            z_after = np.polyfit(after_data['minute'], after_data['momentum'], 1)
            post_slope = z_after[0]
            p_after = np.poly1d(z_after)
            ss_res = np.sum((after_data['momentum'] - p_after(after_data['minute']))**2)
            ss_tot = np.sum((after_data['momentum'] - after_data['momentum'].mean())**2)
            post_r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        if pre_slope is not None and post_slope is not None:
            trend_change = post_slope - pre_slope
        
        windows.append({
            'match_id': match_id,
            'home_team': home_team,
            'away_team': away_team,
            'hydration_minute': hyd_minute,
            'pre_slope': pre_slope,
            'post_slope': post_slope,
            'pre_r2': pre_r2,
            'post_r2': post_r2,
            'trend_change': trend_change,
            'before_data': before_data,
            'after_data': after_data,
            'full_window': window_data
        })
    
    return windows

# Apply to all matches
all_game_windows = []
for idx, row in merged_data.iterrows():
    game_windows = extract_hydration_windows(row)
    all_game_windows.extend(game_windows)

print(f'\nTotal hydration breaks analyzed: {len(all_game_windows)}')
print(f'Across {len(set([w["match_id"] for w in all_game_windows]))} unique matches')

# Display summary
if all_game_windows:
    summary_df = pd.DataFrame({
        'Match ID': [w['match_id'] for w in all_game_windows],
        'Teams': [f"{w['home_team']} vs {w['away_team']}" for w in all_game_windows],
        'Hydration Min': [w['hydration_minute'] for w in all_game_windows],
        'Pre-Slope': [w['pre_slope'] for w in all_game_windows],
        'Post-Slope': [w['post_slope'] for w in all_game_windows],
        'Trend Change': [w['trend_change'] for w in all_game_windows]
    })
    print('\nHydration Break Summary:')
    print(summary_df.to_string())

## Categorize Trends: Downward vs Upward Pre-Treatment

In [ ]:
# Categorize windows by pre-treatment trend
downward_trend = [w for w in all_game_windows if w['pre_slope'] is not None and w['pre_slope'] < -0.5]
upward_trend = [w for w in all_game_windows if w['pre_slope'] is not None and w['pre_slope'] > 0.5]
neutral_trend = [w for w in all_game_windows if w['pre_slope'] is not None and -0.5 <= w['pre_slope'] <= 0.5]

print(f'Downward pre-treatment trends: {len(downward_trend)}')
print(f'Upward pre-treatment trends: {len(upward_trend)}')
print(f'Neutral pre-treatment trends: {len(neutral_trend)}')

print(f'\nDownward trends - Post-treatment change:')
if downward_trend:
    print(f'  Mean trend change: {np.mean([w["trend_change"] for w in downward_trend]):.4f}')
    print(f'  Std trend change: {np.std([w["trend_change"] for w in downward_trend]):.4f}')

print(f'\nUpward trends - Post-treatment change:')
if upward_trend:
    print(f'  Mean trend change: {np.mean([w["trend_change"] for w in upward_trend]):.4f}')
    print(f'  Std trend change: {np.std([w["trend_change"] for w in upward_trend]):.4f}')

## Visualization 1: Overlay All Downward Pre-Treatment Trends

In [ ]:
# Create overlay plot for downward pre-treatment trends
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: All downward trends overlaid (pre-treatment)
for window in downward_trend:
    before_data = window['before_data']
    # Normalize x-axis: 0 = hydration time
    x_normalized = before_data['minute'] - window['hydration_minute']
    ax1.plot(x_normalized, before_data['momentum'], 
             alpha=0.4, linewidth=1.5, color='blue')
    
# Add mean trend line
if downward_trend:
    all_x = np.concatenate([w['before_data']['minute'].values - w['hydration_minute'] 
                            for w in downward_trend])
    all_y = np.concatenate([w['before_data']['momentum'].values for w in downward_trend])
    
    z_mean = np.polyfit(all_x, all_y, 1)
    p_mean = np.poly1d(z_mean)
    x_line = np.linspace(all_x.min(), 0, 100)
    ax1.plot(x_line, p_mean(x_line), 'b-', linewidth=3, label=f'Mean Trend (slope: {z_mean[0]:.3f})')

ax1.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Hydration Break')
ax1.set_xlabel('Minutes from Hydration Break', fontsize=11)
ax1.set_ylabel('Momentum', fontsize=11)
ax1.set_title(f'Pre-Treatment Downward Trends (n={len(downward_trend)})', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: All downward trends overlaid (post-treatment)
for window in downward_trend:
    after_data = window['after_data']
    x_normalized = after_data['minute'] - window['hydration_minute']
    ax2.plot(x_normalized, after_data['momentum'], 
             alpha=0.4, linewidth=1.5, color='red')

# Add mean trend line
if downward_trend:
    all_x = np.concatenate([w['after_data']['minute'].values - w['hydration_minute'] 
                            for w in downward_trend])
    all_y = np.concatenate([w['after_data']['momentum'].values for w in downward_trend])
    
    if len(all_x) > 1:
        z_mean = np.polyfit(all_x, all_y, 1)
        p_mean = np.poly1d(z_mean)
        x_line = np.linspace(0, all_x.max(), 100)
        ax2.plot(x_line, p_mean(x_line), 'r-', linewidth=3, label=f'Mean Trend (slope: {z_mean[0]:.3f})')

ax2.axvline(x=0, color='black', linestyle='--', linewidth=2, label='Hydration Break')
ax2.set_xlabel('Minutes from Hydration Break', fontsize=11)
ax2.set_ylabel('Momentum', fontsize=11)
ax2.set_title(f'Post-Treatment Downward Trends (n={len(downward_trend)})', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('downward_trends_overlay.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: downward_trends_overlay.png')

## Visualization 2: Overlay All Upward Pre-Treatment Trends

In [ ]:
# Create overlay plot for upward pre-treatment trends
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: All upward trends overlaid (pre-treatment)
for window in upward_trend:
    before_data = window['before_data']
    x_normalized = before_data['minute'] - window['hydration_minute']
    ax1.plot(x_normalized, before_data['momentum'], 
             alpha=0.4, linewidth=1.5, color='green')

# Add mean trend line
if upward_trend:
    all_x = np.concatenate([w['before_data']['minute'].values - w['hydration_minute'] 
                            for w in upward_trend])
    all_y = np.concatenate([w['before_data']['momentum'].values for w in upward_trend])
    
    z_mean = np.polyfit(all_x, all_y, 1)
    p_mean = np.poly1d(z_mean)
    x_line = np.linspace(all_x.min(), 0, 100)
    ax1.plot(x_line, p_mean(x_line), 'g-', linewidth=3, label=f'Mean Trend (slope: {z_mean[0]:.3f})')

ax1.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Hydration Break')
ax1.set_xlabel('Minutes from Hydration Break', fontsize=11)
ax1.set_ylabel('Momentum', fontsize=11)
ax1.set_title(f'Pre-Treatment Upward Trends (n={len(upward_trend)})', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: All upward trends overlaid (post-treatment)
for window in upward_trend:
    after_data = window['after_data']
    x_normalized = after_data['minute'] - window['hydration_minute']
    ax2.plot(x_normalized, after_data['momentum'], 
             alpha=0.4, linewidth=1.5, color='orange')

# Add mean trend line
if upward_trend:
    all_x = np.concatenate([w['after_data']['minute'].values - w['hydration_minute'] 
                            for w in upward_trend])
    all_y = np.concatenate([w['after_data']['momentum'].values for w in upward_trend])
    
    if len(all_x) > 1:
        z_mean = np.polyfit(all_x, all_y, 1)
        p_mean = np.poly1d(z_mean)
        x_line = np.linspace(0, all_x.max(), 100)
        ax2.plot(x_line, p_mean(x_line), 'orange', linewidth=3, label=f'Mean Trend (slope: {z_mean[0]:.3f})')

ax2.axvline(x=0, color='black', linestyle='--', linewidth=2, label='Hydration Break')
ax2.set_xlabel('Minutes from Hydration Break', fontsize=11)
ax2.set_ylabel('Momentum', fontsize=11)
ax2.set_title(f'Post-Treatment Upward Trends (n={len(upward_trend)})', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('upward_trends_overlay.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: upward_trends_overlay.png')

## Visualization 3: Direct Before/After Comparison by Trend Type

In [ ]:
# Side-by-side comparison: before and after for both trend types
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Row 1: Downward trends
# Before
ax = axes[0, 0]
for window in downward_trend:
    before_data = window['before_data']
    x_norm = before_data['minute'] - window['hydration_minute']
    ax.plot(x_norm, before_data['momentum'], alpha=0.3, linewidth=1, color='blue')
    ax.scatter(x_norm, before_data['momentum'], alpha=0.2, s=20, color='blue')

if downward_trend:
    all_x = np.concatenate([w['before_data']['minute'].values - w['hydration_minute'] for w in downward_trend])
    all_y = np.concatenate([w['before_data']['momentum'].values for w in downward_trend])
    z = np.polyfit(all_x, all_y, 1)
    p = np.poly1d(z)
    x_line = np.linspace(all_x.min(), 0, 100)
    ax.plot(x_line, p(x_line), 'b-', linewidth=3, label=f'Slope: {z[0]:.3f}')

ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_title('DOWNWARD Pre-Treatment Trends', fontsize=12, fontweight='bold')
ax.set_ylabel('Momentum', fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3)

# After
ax = axes[0, 1]
for window in downward_trend:
    after_data = window['after_data']
    x_norm = after_data['minute'] - window['hydration_minute']
    ax.plot(x_norm, after_data['momentum'], alpha=0.3, linewidth=1, color='red')
    ax.scatter(x_norm, after_data['momentum'], alpha=0.2, s=20, color='red')

if downward_trend:
    all_x = np.concatenate([w['after_data']['minute'].values - w['hydration_minute'] for w in downward_trend])
    all_y = np.concatenate([w['after_data']['momentum'].values for w in downward_trend])
    if len(all_x) > 1:
        z = np.polyfit(all_x, all_y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(0, all_x.max(), 100)
        ax.plot(x_line, p(x_line), 'r-', linewidth=3, label=f'Slope: {z[0]:.3f}')

ax.axvline(x=0, color='black', linestyle='--', linewidth=2)
ax.set_title('DOWNWARD Post-Treatment Trends', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Row 2: Upward trends
# Before
ax = axes[1, 0]
for window in upward_trend:
    before_data = window['before_data']
    x_norm = before_data['minute'] - window['hydration_minute']
    ax.plot(x_norm, before_data['momentum'], alpha=0.3, linewidth=1, color='green')
    ax.scatter(x_norm, before_data['momentum'], alpha=0.2, s=20, color='green')

if upward_trend:
    all_x = np.concatenate([w['before_data']['minute'].values - w['hydration_minute'] for w in upward_trend])
    all_y = np.concatenate([w['before_data']['momentum'].values for w in upward_trend])
    z = np.polyfit(all_x, all_y, 1)
    p = np.poly1d(z)
    x_line = np.linspace(all_x.min(), 0, 100)
    ax.plot(x_line, p(x_line), 'g-', linewidth=3, label=f'Slope: {z[0]:.3f}')

ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_title('UPWARD Pre-Treatment Trends', fontsize=12, fontweight='bold')
ax.set_xlabel('Minutes from Hydration Break', fontsize=10)
ax.set_ylabel('Momentum', fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3)

# After
ax = axes[1, 1]
for window in upward_trend:
    after_data = window['after_data']
    x_norm = after_data['minute'] - window['hydration_minute']
    ax.plot(x_norm, after_data['momentum'], alpha=0.3, linewidth=1, color='orange')
    ax.scatter(x_norm, after_data['momentum'], alpha=0.2, s=20, color='orange')

if upward_trend:
    all_x = np.concatenate([w['after_data']['minute'].values - w['hydration_minute'] for w in upward_trend])
    all_y = np.concatenate([w['after_data']['momentum'].values for w in upward_trend])
    if len(all_x) > 1:
        z = np.polyfit(all_x, all_y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(0, all_x.max(), 100)
        ax.plot(x_line, p(x_line), 'orange', linewidth=3, label=f'Slope: {z[0]:.3f}')

ax.axvline(x=0, color='black', linestyle='--', linewidth=2)
ax.set_title('UPWARD Post-Treatment Trends', fontsize=12, fontweight='bold')
ax.set_xlabel('Minutes from Hydration Break', fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('trend_comparison_grid.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: trend_comparison_grid.png')

## Visualization 4: Individual Game Profiles

In [ ]:
# Create individual plots for each game with multiple hydration breaks
# Group by match_id
games_with_breaks = {}
for window in all_game_windows:
    match_id = window['match_id']
    if match_id not in games_with_breaks:
        games_with_breaks[match_id] = []
    games_with_breaks[match_id].append(window)

# Plot each game
num_games = len(games_with_breaks)
cols = 3
rows = (num_games + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, 5*rows))
axes = axes.flatten() if num_games > 1 else [axes]

for idx, (match_id, windows) in enumerate(games_with_breaks.items()):
    ax = axes[idx]
    
    game_info = next((w for w in windows), None)
    if not game_info:
        continue
    
    title = f"{game_info['home_team']} vs {game_info['away_team']}"
    colors = plt.cm.Set1(np.linspace(0, 1, len(windows)))
    
    for window_idx, window in enumerate(windows):
        color = colors[window_idx]
        hyd_minute = window['hydration_minute']
        
        # Plot full window
        full_window = window['full_window']
        ax.scatter(full_window['minute'], full_window['momentum'], 
                  alpha=0.4, s=30, color=color, label=f'Break @ min {hyd_minute:.0f}')
        
        # Plot pre/post trend lines
        before_data = window['before_data']
        after_data = window['after_data']
        
        if len(before_data) >= 2:
            z = np.polyfit(before_data['minute'], before_data['momentum'], 1)
            p = np.poly1d(z)
            x_line = np.linspace(before_data['minute'].min(), hyd_minute, 50)
            ax.plot(x_line, p(x_line), color=color, linewidth=2.5, linestyle='--', alpha=0.8)
        
        if len(after_data) >= 2:
            z = np.polyfit(after_data['minute'], after_data['momentum'], 1)
            p = np.poly1d(z)
            x_line = np.linspace(hyd_minute, after_data['minute'].max(), 50)
            ax.plot(x_line, p(x_line), color=color, linewidth=2.5, alpha=0.8)
        
        # Mark hydration break
        ax.axvline(x=hyd_minute, color=color, linestyle=':', alpha=0.6)
    
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Match Minute', fontsize=9)
    ax.set_ylabel('Momentum', fontsize=9)
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(len(games_with_breaks), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig('individual_games_profile.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'✓ Saved: individual_games_profile.png (showing {len(games_with_breaks)} games)')

## Visualization 5: Slope Change Distribution

In [ ]:
# Visualize the distribution of slope changes
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Extract trend changes by category
down_trend_changes = [w['trend_change'] for w in downward_trend if w['trend_change'] is not None]
up_trend_changes = [w['trend_change'] for w in upward_trend if w['trend_change'] is not None]

# Plot 1: Histograms
ax = axes[0, 0]
if down_trend_changes:
    ax.hist(down_trend_changes, bins=8, alpha=0.6, label=f'Downward (n={len(down_trend_changes)})', 
            color='blue', edgecolor='black')
if up_trend_changes:
    ax.hist(up_trend_changes, bins=8, alpha=0.6, label=f'Upward (n={len(up_trend_changes)})', 
            color='green', edgecolor='black')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No Change')
ax.set_xlabel('Slope Change (Post - Pre)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Distribution of Slope Changes', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Box plots
ax = axes[0, 1]
data_to_plot = []
labels_to_plot = []
if down_trend_changes:
    data_to_plot.append(down_trend_changes)
    labels_to_plot.append(f'Downward\n(n={len(down_trend_changes)})')
if up_trend_changes:
    data_to_plot.append(up_trend_changes)
    labels_to_plot.append(f'Upward\n(n={len(up_trend_changes)})')

if data_to_plot:
    bp = ax.boxplot(data_to_plot, labels=labels_to_plot, patch_artist=True)
    for patch, color in zip(bp['boxes'], ['lightblue', 'lightgreen']):
        patch.set_facecolor(color)
ax.axhline(y=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.set_ylabel('Slope Change', fontsize=11)
ax.set_title('Slope Change by Pre-Treatment Trend Type', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Scatter - Pre slope vs Trend change
ax = axes[1, 0]
if downward_trend:
    pre_slopes_down = [w['pre_slope'] for w in downward_trend if w['trend_change'] is not None]
    trend_changes_down = [w['trend_change'] for w in downward_trend if w['trend_change'] is not None]
    ax.scatter(pre_slopes_down, trend_changes_down, alpha=0.6, s=100, 
              color='blue', edgecolors='black', label='Downward', marker='o')

if upward_trend:
    pre_slopes_up = [w['pre_slope'] for w in upward_trend if w['trend_change'] is not None]
    trend_changes_up = [w['trend_change'] for w in upward_trend if w['trend_change'] is not None]
    ax.scatter(pre_slopes_up, trend_changes_up, alpha=0.6, s=100, 
              color='green', edgecolors='black', label='Upward', marker='^')

ax.axhline(y=0, color='red', linestyle='--', linewidth=2, alpha=0.5)
ax.axvline(x=0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.set_xlabel('Pre-Treatment Slope', fontsize=11)
ax.set_ylabel('Trend Change (Post - Pre)', fontsize=11)
ax.set_title('Pre-Treatment Slope vs Trend Change', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 4: Summary statistics table
ax = axes[1, 1]
ax.axis('off')

summary_text = "SUMMARY STATISTICS\n" + "="*50 + "\n\n"

if down_trend_changes:
    summary_text += f"DOWNWARD Pre-Treatment Trends (n={len(down_trend_changes)}):\n"
    summary_text += f"  Mean Slope Change: {np.mean(down_trend_changes):.4f}\n"
    summary_text += f"  Std Dev: {np.std(down_trend_changes):.4f}\n"
    summary_text += f"  Min: {np.min(down_trend_changes):.4f}\n"
    summary_text += f"  Max: {np.max(down_trend_changes):.4f}\n"
    summary_text += f"  Median: {np.median(down_trend_changes):.4f}\n"
    # Test if different from 0
    t_stat, p_val = stats.ttest_1samp(down_trend_changes, 0)
    summary_text += f"  T-test vs 0: t={t_stat:.4f}, p={p_val:.4f}\n\n"
else:
    summary_text += "  (No downward trends)\n\n"

if up_trend_changes:
    summary_text += f"UPWARD Pre-Treatment Trends (n={len(up_trend_changes)}):\n"
    summary_text += f"  Mean Slope Change: {np.mean(up_trend_changes):.4f}\n"
    summary_text += f"  Std Dev: {np.std(up_trend_changes):.4f}\n"
    summary_text += f"  Min: {np.min(up_trend_changes):.4f}\n"
    summary_text += f"  Max: {np.max(up_trend_changes):.4f}\n"
    summary_text += f"  Median: {np.median(up_trend_changes):.4f}\n"
    # Test if different from 0
    t_stat, p_val = stats.ttest_1samp(up_trend_changes, 0)
    summary_text += f"  T-test vs 0: t={t_stat:.4f}, p={p_val:.4f}\n"
else:
    summary_text += "  (No upward trends)\n"

ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('slope_change_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: slope_change_analysis.png')

## Summary Statistics and Key Findings

In [ ]:
print("\n" + "="*70)
print("RDD VISUAL ANALYSIS - SUMMARY REPORT")
print("="*70)

print(f"\nTotal hydration breaks analyzed: {len(all_game_windows)}")
print(f"Across {len(set([w['match_id'] for w in all_game_windows]))} unique matches")
print(f"\nDownward pre-treatment trends: {len(downward_trend)}")
print(f"Upward pre-treatment trends: {len(upward_trend)}")
print(f"Neutral pre-treatment trends: {len(neutral_trend)}")

if downward_trend:
    down_changes = [w['trend_change'] for w in downward_trend if w['trend_change'] is not None]
    if down_changes:
        print(f"\nDOWNWARD TRENDS - Post-Hydration Effect:")
        print(f"  Mean trend change: {np.mean(down_changes):+.4f} (std: {np.std(down_changes):.4f})")
        print(f"  Positive changes (improvement): {sum(1 for x in down_changes if x > 0)}/{len(down_changes)}")
        print(f"  Negative changes (worsening): {sum(1 for x in down_changes if x < 0)}/{len(down_changes)}")
        print(f"  Interpretation: Downward momentum trends tend to {'improve' if np.mean(down_changes) > 0 else 'worsen'} after hydration break")

if upward_trend:
    up_changes = [w['trend_change'] for w in upward_trend if w['trend_change'] is not None]
    if up_changes:
        print(f"\nUPWARD TRENDS - Post-Hydration Effect:")
        print(f"  Mean trend change: {np.mean(up_changes):+.4f} (std: {np.std(up_changes):.4f})")
        print(f"  Positive changes (sustained): {sum(1 for x in up_changes if x > 0)}/{len(up_changes)}")
        print(f"  Negative changes (loss of momentum): {sum(1 for x in up_changes if x < 0)}/{len(up_changes)}")
        print(f"  Interpretation: Upward momentum trends tend to {'accelerate' if np.mean(up_changes) > 0 else 'slow'} after hydration break")

print(f"\n" + "="*70)